In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from dataclasses import dataclass, field
from IPython.display import HTML

In [ ]:
@dataclass
class Body:
    name: str
    mass: float
    position: np.ndarray
    velocity: np.ndarray
    color: str = "white"
    size: float = 8.0
    acceleration: np.ndarray = field(default_factory=lambda: np.zeros(2, dtype=float))
    trail: list = field(default_factory=list)

    def __post_init__(self):
        self.position = np.array(self.position, dtype=float)
        self.velocity = np.array(self.velocity, dtype=float)
        self.acceleration = np.array(self.acceleration, dtype=float)

    def record_position(self):
        self.trail.append(self.position.copy())

In [ ]:
def compute_accelerations(bodies, G=1.0, softening=1e-2):
    n = len(bodies)
    accelerations = [np.zeros(2, dtype=float) for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            displacement = bodies[j].position - bodies[i].position
            distance_sq = np.dot(displacement, displacement) + softening**2
            distance = np.sqrt(distance_sq)

            accelerations[i] += G * bodies[j].mass * displacement / (distance_sq * distance)

    return accelerations

In [ ]:
def step_semi_implicit_euler(bodies, dt, G=1.0, softening=1e-2):
    accelerations = compute_accelerations(bodies, G=G, softening=softening)

    for body, acc in zip(bodies, accelerations):
        body.acceleration = acc
        body.velocity = body.velocity + body.acceleration * dt
        body.position = body.position + body.velocity * dt
        body.record_position()

In [ ]:
class Simulation:
    def __init__(self, bodies, dt=0.001, G=1.0, softening=1e-3):
        self.bodies = bodies
        self.dt = dt
        self.G = G
        self.softening = softening
        self.time = 0.0

        for body in self.bodies:
            body.record_position()

    def step(self):
        step_semi_implicit_euler(
            self.bodies,
            dt=self.dt,
            G=self.G,
            softening=self.softening
        )
        self.time += self.dt

    def run(self, steps):
        for _ in range(steps):
            self.step()

In [ ]:
def make_two_body_system():
    sun = Body(
        name="Sun",
        mass=1000.0,
        position=[0.0, 0.0],
        velocity=[0.0, 0.0],
        color="gold",
        size=18
    )

    earth = Body(
        name="Earth",
        mass=1.0,
        position=[1.0, 0.0],
        velocity=[0.0, 31.6],
        color="deepskyblue",
        size=8
    )

    return [sun, earth]

In [ ]:
bodies = make_two_body_system()
sim = Simulation(bodies, dt=0.0005, G=1.0, softening=1e-3)
sim.run(4000)

In [ ]:
plt.figure(figsize=(8, 8))
ax = plt.gca()
ax.set_facecolor("black")

for body in sim.bodies:
    trail = np.array(body.trail)
    plt.plot(trail[:, 0], trail[:, 1], color=body.color, alpha=0.7, linewidth=1.5)
    plt.scatter(body.position[0], body.position[1], color=body.color, s=body.size * 15, label=body.name)

plt.legend()
plt.title("N-Body Orbital Simulation", color="white")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
def animate_simulation(bodies, dt=0.0005, steps=2000, G=1.0, softening=1e-3, trail_length=300):
    sim = Simulation(bodies, dt=dt, G=G, softening=softening)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_facecolor("black")
    fig.patch.set_facecolor("black")

    max_range = 2.5
    ax.set_xlim(-max_range, max_range)
    ax.set_ylim(-max_range, max_range)
    ax.set_aspect("equal")
    ax.set_title("N-Body Orbital Physics Simulator", color="white")
    ax.tick_params(colors="white")

    scatters = []
    trail_lines = []

    for body in sim.bodies:
        scatter = ax.scatter([], [], color=body.color, s=body.size * 15)
        line, = ax.plot([], [], color=body.color, alpha=0.6, linewidth=1.2)
        scatters.append(scatter)
        trail_lines.append(line)

    def init():
        for scatter, line in zip(scatters, trail_lines):
            scatter.set_offsets(np.array([[np.nan, np.nan]]))
            line.set_data([], [])
        return scatters + trail_lines

    def update(frame):
        sim.step()

        for i, body in enumerate(sim.bodies):
            scatters[i].set_offsets(body.position.reshape(1, 2))

            trail = np.array(body.trail[-trail_length:])
            if len(trail) > 1:
                trail_lines[i].set_data(trail[:, 0], trail[:, 1])

        return scatters + trail_lines

    anim = FuncAnimation(
        fig,
        update,
        frames=steps,
        init_func=init,
        interval=20,
        blit=True
    )

    plt.close(fig)
    return anim

In [ ]:
bodies = make_two_body_system()
anim = animate_simulation(
    bodies=bodies,
    dt=0.0005,
    steps=1500,
    G=1.0,
    softening=1e-3,
    trail_length=250
)

HTML(anim.to_jshtml())

In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100

In [ ]:
bodies = make_two_body_system()
anim = animate_simulation(
    bodies=bodies,
    dt=0.0005,
    steps=1500,
    G=1.0,
    softening=1e-3,
    trail_length=250
)

HTML(anim.to_jshtml())

In [ ]:
import matplotlib as mpl

# increase the allowed animation size (MB)
mpl.rcParams["animation.embed_limit"] = 100

In [ ]:
bodies = make_two_body_system()
anim = animate_simulation(
    bodies=bodies,
    dt=0.0005,
    steps=1500,
    G=1.0,
    softening=1e-3,
    trail_length=250
)

HTML(anim.to_jshtml())

In [ ]:
def step_velocity_verlet(bodies, dt, G=1.0, softening=1e-3):
    # 1) compute current accelerations a(t)
    old_accelerations = compute_accelerations(bodies, G=G, softening=softening)

    # 2) update positions using:
    #    r(t+dt) = r(t) + v(t)dt + 0.5*a(t)*dt^2
    for body, acc in zip(bodies, old_accelerations):
        body.acceleration = acc
        body.position = body.position + body.velocity * dt + 0.5 * acc * dt**2

    # 3) compute new accelerations a(t+dt) from the updated positions
    new_accelerations = compute_accelerations(bodies, G=G, softening=softening)

    # 4) update velocities using:
    #    v(t+dt) = v(t) + 0.5*(a(t) + a(t+dt))*dt
    for body, old_acc, new_acc in zip(bodies, old_accelerations, new_accelerations):
        body.velocity = body.velocity + 0.5 * (old_acc + new_acc) * dt
        body.acceleration = new_acc
        body.record_position()

In [ ]:
class SimulationVV:
    def __init__(self, bodies, dt=0.001, G=1.0, softening=1e-3):
        self.bodies = bodies
        self.dt = dt
        self.G = G
        self.softening = softening
        self.time = 0.0

        for body in self.bodies:
            if len(body.trail) == 0:
                body.record_position()

    def step(self):
        step_velocity_verlet(
            self.bodies,
            dt=self.dt,
            G=self.G,
            softening=self.softening
        )
        self.time += self.dt

    def run(self, steps):
        for _ in range(steps):
            self.step()

In [ ]:
def animate_simulation_vv(bodies, dt=0.0005, steps=1200, G=1.0, softening=1e-3, trail_length=250):
    sim = SimulationVV(bodies, dt=dt, G=G, softening=softening)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_facecolor("black")
    fig.patch.set_facecolor("black")

    max_range = 2.5
    ax.set_xlim(-max_range, max_range)
    ax.set_ylim(-max_range, max_range)
    ax.set_aspect("equal")
    ax.set_title("N-Body Orbital Physics Simulator (Velocity Verlet)", color="white")
    ax.tick_params(colors="white")

    scatters = []
    trail_lines = []

    for body in sim.bodies:
        scatter = ax.scatter([], [], color=body.color, s=body.size * 15)
        line, = ax.plot([], [], color=body.color, alpha=0.6, linewidth=1.2)
        scatters.append(scatter)
        trail_lines.append(line)

    def init():
        for scatter, line in zip(scatters, trail_lines):
            scatter.set_offsets(np.array([[np.nan, np.nan]]))
            line.set_data([], [])
        return scatters + trail_lines

    def update(frame):
        sim.step()

        for i, body in enumerate(sim.bodies):
            scatters[i].set_offsets(body.position.reshape(1, 2))
            trail = np.array(body.trail[-trail_length:])
            if len(trail) > 1:
                trail_lines[i].set_data(trail[:, 0], trail[:, 1])

        return scatters + trail_lines

    anim = FuncAnimation(
        fig,
        update,
        frames=steps,
        init_func=init,
        interval=20,
        blit=True
    )

    plt.close(fig)
    return anim

In [ ]:
bodies = make_better_two_body_system()

anim = animate_simulation_vv(
    bodies=bodies,
    dt=0.0005,
    steps=1000,
    G=1.0,
    softening=1e-3,
    trail_length=250
)

HTML(anim.to_jshtml())

In [ ]:
def make_better_two_body_system():
    M = 1000.0
    r = 1.0
    G = 1.0
    v = np.sqrt(G * M / r)

    sun = Body(
        name="Sun",
        mass=M,
        position=[0.0, 0.0],
        velocity=[0.0, 0.0],
        color="gold",
        size=18
    )

    earth = Body(
        name="Earth",
        mass=1.0,
        position=[r, 0.0],
        velocity=[0.0, v],
        color="deepskyblue",
        size=8
    )

    return [sun, earth]

In [ ]:
def compute_kinetic_energy(bodies):
    total_ke = 0.0
    for body in bodies:
        total_ke += 0.5 * body.mass * np.dot(body.velocity, body.velocity)
    return total_ke


def compute_potential_energy(bodies, G=1.0, softening=1e-3):
    total_pe = 0.0
    n = len(bodies)

    for i in range(n):
        for j in range(i + 1, n):
            displacement = bodies[j].position - bodies[i].position
            distance = np.sqrt(np.dot(displacement, displacement) + softening**2)
            total_pe += -G * bodies[i].mass * bodies[j].mass / distance

    return total_pe

In [ ]:
def run_with_energy_tracking_vv(bodies, steps=2000, dt=0.0005, G=1.0, softening=1e-3):
    sim = SimulationVV(bodies, dt=dt, G=G, softening=softening)

    kinetic_history = []
    potential_history = []
    total_history = []

    for _ in range(steps):
        sim.step()
        ke = compute_kinetic_energy(sim.bodies)
        pe = compute_potential_energy(sim.bodies, G=G, softening=softening)
        kinetic_history.append(ke)
        potential_history.append(pe)
        total_history.append(ke + pe)

    return sim, kinetic_history, potential_history, total_history

In [ ]:
bodies = make_better_two_body_system()

sim, ke_hist, pe_hist, total_hist = run_with_energy_tracking_vv(
    bodies,
    steps=3000,
    dt=0.0005,
    G=1.0,
    softening=1e-3
)

plt.figure(figsize=(10, 5))
plt.plot(ke_hist, label="Kinetic Energy")
plt.plot(pe_hist, label="Potential Energy")
plt.plot(total_hist, label="Total Energy")
plt.xlabel("Step")
plt.ylabel("Energy")
plt.title("Energy Diagnostics (Velocity Verlet)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def make_three_body_system():
    body1 = Body(
        name="Body 1",
        mass=500.0,
        position=[-0.8, 0.0],
        velocity=[0.0, -10.0],
        color="orange",
        size=14
    )

    body2 = Body(
        name="Body 2",
        mass=500.0,
        position=[0.8, 0.0],
        velocity=[0.0, 10.0],
        color="cyan",
        size=14
    )

    body3 = Body(
        name="Body 3",
        mass=5.0,
        position=[0.0, 1.2],
        velocity=[18.0, 0.0],
        color="magenta",
        size=8
    )

    return [body1, body2, body3]

In [ ]:
bodies = make_three_body_system()

anim = animate_simulation_vv(
    bodies=bodies,
    dt=0.0005,
    steps=1200,
    G=1.0,
    softening=1e-3,
    trail_length=250
)

HTML(anim.to_jshtml())

In [ ]:
bodies = make_three_body_system()

anim = animate_simulation_vv(
    bodies=bodies,
    dt=0.001,
    steps=500,
    G=1.0,
    softening=1e-3,
    trail_length=120
)

HTML(anim.to_jshtml())

In [ ]:
def compute_kinetic_energy(bodies):
    total_ke = 0.0
    for body in bodies:
        total_ke += 0.5 * body.mass * np.dot(body.velocity, body.velocity)
    return total_ke


def compute_potential_energy(bodies, G=1.0, softening=1e-3):
    total_pe = 0.0
    n = len(bodies)

    for i in range(n):
        for j in range(i + 1, n):
            displacement = bodies[j].position - bodies[i].position
            distance = np.sqrt(np.dot(displacement, displacement) + softening**2)
            total_pe += -G * bodies[i].mass * bodies[j].mass / distance

    return total_pe

In [ ]:
def run_with_energy_tracking_vv(bodies, steps=2000, dt=0.0005, G=1.0, softening=1e-3):
    sim = SimulationVV(bodies, dt=dt, G=G, softening=softening)

    kinetic_history = []
    potential_history = []
    total_history = []

    for _ in range(steps):
        sim.step()
        ke = compute_kinetic_energy(sim.bodies)
        pe = compute_potential_energy(sim.bodies, G=G, softening=softening)
        kinetic_history.append(ke)
        potential_history.append(pe)
        total_history.append(ke + pe)

    return sim, kinetic_history, potential_history, total_history

In [ ]:
bodies = make_better_two_body_system()

sim, ke_hist, pe_hist, total_hist = run_with_energy_tracking_vv(
    bodies,
    steps=3000,
    dt=0.0005,
    G=1.0,
    softening=1e-3
)

plt.figure(figsize=(10, 5))
plt.plot(ke_hist, label="Kinetic Energy")
plt.plot(pe_hist, label="Potential Energy")
plt.plot(total_hist, label="Total Energy")
plt.xlabel("Step")
plt.ylabel("Energy")
plt.title("Energy Diagnostics (Velocity Verlet)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def make_three_body_system():
    body1 = Body(
        name="Body 1",
        mass=500.0,
        position=[-0.8, 0.0],
        velocity=[0.0, -10.0],
        color="orange",
        size=14
    )

    body2 = Body(
        name="Body 2",
        mass=500.0,
        position=[0.8, 0.0],
        velocity=[0.0, 10.0],
        color="cyan",
        size=14
    )

    body3 = Body(
        name="Body 3",
        mass=5.0,
        position=[0.0, 1.2],
        velocity=[18.0, 0.0],
        color="magenta",
        size=8
    )

    return [body1, body2, body3]

In [ ]:
bodies = make_three_body_system()

anim = animate_simulation_vv(
    bodies=bodies,
    dt=0.0005,
    steps=1200,
    G=1.0,
    softening=1e-3,
    trail_length=250
)

HTML(anim.to_jshtml())

In [ ]:
def animate_simulation_vv_fast(bodies, dt=0.0005, frames=300, steps_per_frame=4, G=1.0, softening=1e-3, trail_length=100):
    sim = SimulationVV(bodies, dt=dt, G=G, softening=softening)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_facecolor("black")
    fig.patch.set_facecolor("black")

    max_range = 2.5
    ax.set_xlim(-max_range, max_range)
    ax.set_ylim(-max_range, max_range)
    ax.set_aspect("equal")
    ax.set_title("N-Body Orbital Physics Simulator (Fast Velocity Verlet)", color="white")
    ax.tick_params(colors="white")

    scatters = []
    trail_lines = []

    for body in sim.bodies:
        scatter = ax.scatter([], [], color=body.color, s=body.size * 15)
        line, = ax.plot([], [], color=body.color, alpha=0.6, linewidth=1.2)
        scatters.append(scatter)
        trail_lines.append(line)

    def init():
        for scatter, line in zip(scatters, trail_lines):
            scatter.set_offsets(np.array([[np.nan, np.nan]]))
            line.set_data([], [])
        return scatters + trail_lines

    def update(frame):
        for _ in range(steps_per_frame):
            sim.step()

        for i, body in enumerate(sim.bodies):
            scatters[i].set_offsets(body.position.reshape(1, 2))
            trail = np.array(body.trail[-trail_length:])
            if len(trail) > 1:
                trail_lines[i].set_data(trail[:, 0], trail[:, 1])

        return scatters + trail_lines

    anim = FuncAnimation(
        fig,
        update,
        frames=frames,
        init_func=init,
        interval=30,
        blit=True
    )

    plt.close(fig)
    return anim

In [ ]:
bodies = make_three_body_system()

anim_fast = animate_simulation_vv_fast(
    bodies=bodies,
    dt=0.0005,
    frames=300,
    steps_per_frame=4,
    G=1.0,
    softening=1e-3,
    trail_length=100
)

HTML(anim_fast.to_jshtml())

In [ ]:
bodies = make_three_body_system()

anim = animate_simulation_vv(
    bodies=bodies,
    dt=0.0005,
    steps=400,
    G=1.0,
    softening=1e-3,
    trail_length=120
)

anim.save("three_body_demo.gif", writer="pillow", fps=20)